# Stage 4 — Frozen three-seed final validation run

Runs seeds 42, 43, and 44 from fresh pinned E5-small-v2 weights using the frozen patience-2 hard-negative configuration. All seeds reuse the exact same approved BM25+dense mined-negative artifact. Validation only: test files are never loaded. After training, the notebook computes 95% percentile-bootstrap confidence intervals and requires distinct model, embedding, and loss hashes before reporting success.


In [ ]:
%pip install -q "sentence-transformers==6.0.1" "transformers==5.16.1" "faiss-cpu==1.15.0" "rank-bm25==0.2.2" datasets pandas


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import base64
import hashlib
import io
import zipfile
from pathlib import Path

PAYLOAD_SHA256 = '572F87BC42D9FA52D68BE661286E2F8FDA16563FA75B9E5033A3C2C89BB46F19'
PAYLOAD_B64 = '...'
payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest().upper() == PAYLOAD_SHA256
SOURCE_DIR = Path('/content/drive/MyDrive/RAFT/stage4_hardneg/final_source_snapshot')
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(SOURCE_DIR)
print(f'STAGE4_FINAL_SOURCE_READY={SOURCE_DIR}')
print(f'PAYLOAD_SHA256={PAYLOAD_SHA256}')


In [ ]:
import copy
import json
import subprocess
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/RAFT')
PILOT_CONFIG = json.loads((SOURCE_DIR / 'configs/stage4_hardneg_pilot.json').read_text())
FINAL_CONFIGS = [json.loads((SOURCE_DIR / f'configs/stage4_hardneg_final_seed{seed}.json').read_text()) for seed in (42, 43, 44)]
required = [
    DRIVE_ROOT / 'stage1_scifact/processed/corpus.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/queries_train.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/qrels_train.tsv',
    DRIVE_ROOT / 'stage1_scifact/processed/queries_validation.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/qrels_validation.tsv',
    DRIVE_ROOT / 'stage2_baselines/results/stage2_baselines.csv',
    DRIVE_ROOT / 'stage3_inbatch/final_results/stage3_final_summary.json',
    DRIVE_ROOT / 'stage4_hardneg/mined/stage4_seed42_hard_negatives.jsonl',
    DRIVE_ROOT / 'stage4_hardneg/mined/stage4_seed42_mining_summary.json',
    DRIVE_ROOT / 'stage4_hardneg/smoke_seed42/results/stage4_smoke_result.json',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing required Drive artifacts: {missing}'
assert torch.cuda.is_available(), 'Select a Colab GPU runtime before running Stage 4 final.'
assert [c['training']['random_seed'] for c in FINAL_CONFIGS] == [42, 43, 44]
for config in FINAL_CONFIGS:
    assert config['model'] == PILOT_CONFIG['model']
    assert config['dataset'] == PILOT_CONFIG['dataset']
    assert config['mining'] == PILOT_CONFIG['mining']
    assert config['training']['early_stopping_rule']['patience_evaluations'] == 2
    pilot_training = copy.deepcopy(PILOT_CONFIG['training'])
    final_training = copy.deepcopy(config['training'])
    pilot_training.pop('random_seed')
    final_training.pop('random_seed')
    assert pilot_training == final_training
    assert config.get('pilot_variant') is None
print(f'GPU={torch.cuda.get_device_name(0)}')
print('STAGE4_FROZEN_PATIENCE2_THREE_SEED_PREFLIGHT_PASSED')


In [ ]:
subprocess.run([
    sys.executable,
    str(SOURCE_DIR / 'src/run_stage4_final.py'),
    '--repo-dir', str(SOURCE_DIR),
    '--drive-root', str(DRIVE_ROOT),
], check=True)


In [ ]:
import pandas as pd

RESULTS_DIR = DRIVE_ROOT / 'stage4_hardneg/final_results'
SUMMARY_PATH = RESULTS_DIR / 'stage4_final_summary.json'
AUDIT_PATH = RESULTS_DIR / 'stage4_final_seed_audit.json'
summary = json.loads(SUMMARY_PATH.read_text())
audit = json.loads(AUDIT_PATH.read_text())
assert summary['random_seeds'] == [42, 43, 44]
assert summary['early_stopping_patience_evaluations'] == 2
assert summary['test_split_loaded'] is False
assert audit['all_model_hashes_distinct'] is True
assert audit['all_embedding_hashes_distinct'] is True
assert audit['all_loss_hashes_distinct'] is True
assert audit['test_split_loaded'] is False
print('PER-SEED VALIDATION RESULTS:')
display(pd.read_csv(RESULTS_DIR / 'stage4_final_per_seed_validation.csv'))
print('\nAGGREGATED MEAN AND 95% BOOTSTRAP CI:')
display(pd.read_csv(RESULTS_DIR / 'stage4_final_aggregate_validation.csv'))
audit_table = pd.read_csv(RESULTS_DIR / 'stage4_final_seed_audit.csv')
for column in ('result_sha256', 'loss_sha256', 'embedding_sha256', 'model_sha256'):
    audit_table[column] = audit_table[column].str[:16]
print('\nCHECKSUM/DISTINCTNESS AUDIT (SHA-256 prefixes shown):')
display(audit_table)
print('\nPAIRWISE EMBEDDING AUDIT:')
print(json.dumps(audit['pairwise_embedding_comparisons'], indent=2, sort_keys=True))
print(f"\nTEST SPLIT LOADED: {summary['test_split_loaded']}")
print(f'Attach these files in the Codex task: {SUMMARY_PATH} and {AUDIT_PATH}')
